# RelCheck: Fine-Tune Spatial Relation Verifier

Fine-tunes LLaVA-1.5-7B with LoRA on the **VSR (Visual Spatial Reasoning)** dataset.

- **Dataset**: [juletxara/visual-spatial-reasoning](https://huggingface.co/datasets/juletxara/visual-spatial-reasoning) — 10k+ image-caption pairs with True/False spatial relation labels
- **Model**: LLaVA-1.5-7B (llava-hf/llava-1.5-7b-hf) with LoRA r=8
- **Task**: Given image + claim → TRUE or FALSE
- **Goal**: Replace zero-shot Maverick VQA with a specialized spatial verifier

**Expected training time**: ~2-3 hrs on Colab A100  
**Expected improvement**: Reduce false positives in spatial triple verification

## Cells
1. Setup
2. Load VSR dataset
3. Load LLaVA-1.5-7B + LoRA config
4. Training
5. Evaluate on VSR test set
6. Save to Drive

In [ ]:
# ============================================================
# Cell 1 -- Setup
# ============================================================
!pip install -q transformers datasets peft accelerate bitsandbytes>=0.46.1 trl pillow torch torchvision

from google.colab import drive
drive.mount('/content/drive')

import os, json, random, torch
import numpy as np
from pathlib import Path
from PIL import Image
import requests
from io import BytesIO

SAVE_DIR = '/content/drive/MyDrive/RelCheck'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_OUT_DIR = f'{SAVE_DIR}/verifier_lora'
os.makedirs(MODEL_OUT_DIR, exist_ok=True)

print('Setup complete.')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ============================================================
# Cell 2 -- Load VSR Dataset
# ============================================================
# VSR: Visual Spatial Reasoning -- 10k+ True/False spatial relation pairs
# Format: {image (PIL), caption, label (True/False)}
# Caption describes spatial relation between two objects.
# Label = True means caption correctly describes image.

from datasets import load_dataset

# Load random split (train/dev/test split by image)
vsr = load_dataset('juletxara/visual-spatial-reasoning', 'random')

train_data = vsr['train']
val_data   = vsr['dev']
test_data  = vsr['test']

print(f'Train: {len(train_data)} examples')
print(f'Val:   {len(val_data)} examples')
print(f'Test:  {len(test_data)} examples')
print(f'\nSample entry:')
sample = train_data[0]
print(f'  caption: {sample["caption"]}')
print(f'  label:   {sample["label"]}')
print(f'  image:   {type(sample["image"])}')

In [ ]:
# ============================================================
# Cell 3 -- Load LLaVA-1.5-7B + LoRA Config
# ============================================================
from transformers import LlavaForConditionalGeneration, LlavaProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

# 4-bit quantization to fit in Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = LlavaProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'right'

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

# LoRA config -- target attention layers
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% trainable parameters

In [ ]:
# ============================================================
# Cell 4 -- Dataset Collator + Training
# ============================================================
from torch.utils.data import Dataset, DataLoader
from transformers import TrainingArguments, Trainer
from trl import SFTTrainer

VERIFY_PROMPT_TEMPLATE = (
    'USER: <image>\n'
    'Look carefully at this image.\n'
    'Claim: "{caption}"\n'
    'Is this claim TRUE or FALSE based on what you see?\n'
    'Answer with TRUE or FALSE.\n'
    'ASSISTANT:'
)

class VSRDataset(Dataset):
    def __init__(self, hf_dataset, processor, max_samples=None):
        self.data = hf_dataset
        if max_samples:
            self.data = self.data.select(range(min(max_samples, len(self.data))))
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        caption = item['caption']
        label = 'TRUE' if item['label'] else 'FALSE'

        prompt = VERIFY_PROMPT_TEMPLATE.format(caption=caption)
        full_text = prompt + ' ' + label

        encoding = self.processor(
            images=image,
            text=full_text,
            return_tensors='pt',
            padding='max_length',
            max_length=256,
            truncation=True,
        )
        # Labels: mask the prompt, only train on the TRUE/FALSE token
        labels = encoding['input_ids'].clone()
        prompt_enc = self.processor.tokenizer(
            prompt, return_tensors='pt', max_length=256, truncation=True
        )
        prompt_len = prompt_enc['input_ids'].shape[1]
        labels[0, :prompt_len] = -100  # mask prompt tokens

        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'pixel_values':   encoding['pixel_values'].squeeze(0),
            'labels':         labels.squeeze(0),
        }


train_dataset = VSRDataset(train_data, processor)
val_dataset   = VSRDataset(val_data,   processor, max_samples=500)

print(f'Train examples: {len(train_dataset)}')
print(f'Val examples:   {len(val_dataset)}')

# Training arguments
training_args = TrainingArguments(
    output_dir=MODEL_OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,     # effective batch = 16
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=False,
    bf16=True,
    dataloader_num_workers=2,
    report_to='none',
    remove_unused_columns=False,
)

def data_collator(batch):
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'pixel_values':   torch.stack([b['pixel_values']   for b in batch]),
        'labels':         torch.stack([b['labels']         for b in batch]),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print('Starting training... (~2-3 hours on A100)')
trainer.train()
print('Training complete!')

In [ ]:
# ============================================================
# Cell 5 -- Evaluate on VSR Test Set
# ============================================================
model.eval()

correct = 0
total   = 0
tp = tn = fp = fn = 0

test_sample = test_data.select(range(min(500, len(test_data))))

for item in test_sample:
    image   = item['image'].convert('RGB')
    caption = item['caption']
    gt      = 'TRUE' if item['label'] else 'FALSE'

    prompt = VERIFY_PROMPT_TEMPLATE.format(caption=caption)
    enc = processor(
        images=image, text=prompt,
        return_tensors='pt', padding=True
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=5, do_sample=False)

    decoded = processor.decode(out[0], skip_special_tokens=True)
    pred = 'TRUE' if 'TRUE' in decoded.upper().split('ASSISTANT:')[-1] else 'FALSE'

    total += 1
    if pred == gt:
        correct += 1
    if gt == 'TRUE'  and pred == 'TRUE':  tp += 1
    if gt == 'FALSE' and pred == 'FALSE': tn += 1
    if gt == 'FALSE' and pred == 'TRUE':  fp += 1
    if gt == 'TRUE'  and pred == 'FALSE': fn += 1

print(f'VSR Test Accuracy: {correct/total:.1%} ({correct}/{total})')
print(f'Precision: {tp/(tp+fp+1e-9):.1%}')
print(f'Recall:    {tp/(tp+fn+1e-9):.1%}')
print(f'F1:        {2*tp/(2*tp+fp+fn+1e-9):.1%}')
print(f'False positive rate: {fp/(fp+tn+1e-9):.1%}  (lower = fewer false hallucination detections)')

In [ ]:
# ============================================================
# Cell 6 -- Save LoRA Weights to Drive
# ============================================================
LORA_SAVE_PATH = f'{SAVE_DIR}/verifier_lora_weights'
model.save_pretrained(LORA_SAVE_PATH)
processor.save_pretrained(LORA_SAVE_PATH)
print(f'LoRA weights saved to: {LORA_SAVE_PATH}')
print()
print('To use in RelCheck_600.ipynb:')
print('  1. Load base LLaVA-1.5-7B')
print('  2. from peft import PeftModel')
print(f'  3. model = PeftModel.from_pretrained(base_model, "{LORA_SAVE_PATH}")')
print('  4. Replace verify_with_crop() Maverick call with local model inference')